In [218]:
#!/usr/bin/env python
# coding: utf-8

import math
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
import pandas as pd
from datetime import datetime
import pickle
import re
from pyxdameraulevenshtein import damerau_levenshtein_distance
import apsw
import sys
import numpy as np
import corp_simplify_utils
import seaborn as sns
import matplotlib.pyplot as plt
import pyreadr
from collections import Counter

# nlp
import spacy
from spacy import displacy
from collections import Counter
# to install: $python3 -m spacy download en_core_web_lg
import en_core_web_lg

# analysis/regressions
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from scipy.stats import ks_2samp
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind

# from statsmodels.graphics.gofplots import qqplot_2samples
from scipy import stats
from joypy import joyplot
from matplotlib import cm

from datetime import date
today_for_filenames = date.today()
curr_date = str(today_for_filenames.strftime("%Y%m%d"))


NUMBER_OF_MATCHES_TO_RECORD = 10
punc_remove_re = re.compile(r'\W+')
corp_re = re.compile('( (group|holding(s)?( co)?|inc(orporated)?|ltd|l ?l? ?[cp]|co(rp(oration)?|mpany)?|s[ae]|plc))+$')
and_re = re.compile(' & ')
punc1_re = re.compile(r'(?<=\S)[\'’´\.](?=\S)')
punc2_re = re.compile(r'[\s\.,:;/\'"`´‘’“”\(\)\[\]\{\}_—\-?$=!]+')

STOPWORDS = nltk.corpus.stopwords.words('english')
STOPWORDS.remove("am")
STOPWORDS.remove("up")
STOPWORDS.remove("in")
STOPWORDS.remove("on")
STOPWORDS.remove("all")
STOPWORDS.remove("any")
STOPWORDS.remove("most")
STOPWORDS.remove("no")
STOPWORDS.remove("nor")
STOPWORDS.remove("own")
STOPWORDS.remove("same")
STOPWORDS.remove("so")
STOPWORDS.remove("very")
STOPWORDS.remove("s")
STOPWORDS.remove("t")
STOPWORDS.remove("d")
STOPWORDS.remove("ll")
STOPWORDS.remove("m")
STOPWORDS.remove("o")
STOPWORDS.remove("re")
STOPWORDS.remove("ve")
STOPWORDS.remove("y")

#compile regex patterns to reuse
STOPWORD_RE = re.compile(r'\b(the|of|and|in|on)\b', re.IGNORECASE)
CORP_SUFFIX_RE = re.compile(r'\b(inc|corp|ltd|llc|plc|co|company|limited)\b', re.IGNORECASE)
PDF_PATTERN_RE = re.compile(r'\s[0-9]*\s[km]b\s*pdf', re.IGNORECASE)
PUNCT_RE = re.compile(r'[^\w\s-]')  # match punctuation
MULTISPACE_RE = re.compile(r'\s+')

stopword_re_str = r""
for word in STOPWORDS:
	stopword_re_str += r'\b' + word + r'\b|'
stopword_re = re.compile(stopword_re_str[:-1]) # The negative 1 is for the fencepost |

NON_FINANCIAL_ORG_TERMS = [
    'university', 'college', 'school', 'institute', 'academy', 
    'hospital', 'medical center', 'health system', 'center', 
    'commission', 'authority', 'association', 'society', 
    'foundation', 'transportation services', 'district', 
    'chamber', 'commerce', 'library', 'museum', 'public', 
    'city', 'county', 'town', 'government', 'state', 'federal',
    'ministry', 'department', 'office'
]

NON_FINANCIAL_RE = re.compile(r'\b(' + '|'.join(NON_FINANCIAL_ORG_TERMS) + r')\b', re.IGNORECASE)

BASE_DIR = "/Users/aawesomez/Documents/UROP/NLP-regextable/"
# BASE_DIR = "/Users/jameschen/Team Name Dropbox/James Chen/JLW-FINREG-PARTICIPATION/"
# BASE_DIR = "/Users/jameschen/Documents/Code/JLW-FINREG-PARTICIPATION/"
# DB_PATH = BASE_DIR + "data/master.sqlite"
DB_PATH = BASE_DIR + "Data/master.sqlite"
# LAST_SAVE_DATASET_DATE = "20210824"
LAST_SAVE_DATASET_DATE = "20220402" # Needs to be set to the last date the 'rebuild datasets' part of this code was run

# Function to calculate longest common substring, from https://www.geeksforgeeks.org/print-longest-common-substring/
# function to find and print 
# the longest common substring of
# X[0..m-1] and Y[0..n-1]
def get_longest_common_substring(X, Y, m, n):
 
    # Create a table to store lengths of
    # longest common suffixes of substrings.
    # Note that LCSuff[i][j] contains length
    # of longest common suffix of X[0..i-1] and
    # Y[0..j-1]. The first row and first
    # column entries have no logical meaning,
    # they are used only for simplicity of program
    LCSuff = [[0 for i in range(n + 1)]
                 for j in range(m + 1)]
 
    # To store length of the
    # longest common substring
    length = 0
 
    # To store the index of the cell
    # which contains the maximum value.
    # This cell's index helps in building
    # up the longest common substring
    # from right to left.
    row, col = 0, 0
 
    # Following steps build LCSuff[m+1][n+1]
    # in bottom up fashion.
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0:
                LCSuff[i][j] = 0
            elif X[i - 1] == Y[j - 1]:
                LCSuff[i][j] = LCSuff[i - 1][j - 1] + 1
                if length < LCSuff[i][j]:
                    length = LCSuff[i][j]
                    row = i
                    col = j
            else:
                LCSuff[i][j] = 0
 
    # if true, then no common substring exists
    if length == 0:
        return ""
 
    # allocate space for the longest
    # common substring
    resultStr = ['0'] * length
 
    # traverse up diagonally form the
    # (row, col) cell until LCSuff[row][col] != 0
    while LCSuff[row][col] != 0:
        length -= 1
        resultStr[length] = X[row - 1] # or Y[col-1]
 
        # move diagonally up to previous cell
        row -= 1
        col -= 1
 
    # required longest common substring
    longest_common_substring = ''.join(resultStr)

    return longest_common_substring


# Function from Brad Hackinen's NAMA
def basicHash(s):
    '''
    A simple case and puctuation-insensitive hash
    '''
    s = s.lower()
    s = re.sub(and_re,' and ',s)
    s = re.sub(punc1_re,'',s)
    s = re.sub(punc2_re,' ',s)
    s = s.strip()

    return s

# Function from Brad Hackinen's NAMA
def corpHash(s):
    '''
    A hash function for corporate subsidiaries
    Insensitive to
        -case & punctation
        -'the' prefix
        -common corporation suffixes, including 'holding co'
    '''
    s = basicHash(s)
    if s.startswith('the '):
        s = s[4:]

    s = re.sub(corp_re,'',s,count=1)

    return s

# function to clean org names
def clean_fin_org_names(name: str) -> str:
    if name is None or not isinstance(name, str) or name == "NA":
        return ""
    
    # James strip metadata from name
    name = name.split(',')[0]
    #Remove patterns like "10 kb pdf"
    name = PDF_PATTERN_RE.sub("", name)

    #Unicode and punctuation cleanup
    name = corp_simplify_utils.normalize_unicode(name)
    name = PUNCT_RE.sub(" ", name)

    #Remove corporate suffixes and stopwords and non-financial entity
    name = CORP_SUFFIX_RE.sub("", name)
    name = NON_FINANCIAL_RE.sub("", name)
    name = STOPWORD_RE.sub("", name)

    #Normalize spacing and lowercase
    name = MULTISPACE_RE.sub(" ", name).strip().lower()

    return name

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/stevenkang/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [219]:
# Locate data directory and read in data files
import os
from pathlib import Path

current_dir = Path(os.getcwd()).parent
data_dir = current_dir / 'data'
data_dir = data_dir.resolve()

try:
    compustat_df = pd.read_csv(data_dir / 'CompustatNames.csv')
    cik_df = pd.read_csv(data_dir / 'CIK.csv')
    fdic_df = pd.read_csv(data_dir / 'FDIC_clean.csv') # Using your 'FDIC_clean.csv'
    sec_df = pd.read_csv(data_dir / 'SEC_Institutions.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()
    

In [220]:
# cleaning and standardizing organization names
compustat_df['std_name'] = compustat_df['conm'].apply(clean_fin_org_names)
fdic_df['std_name'] = fdic_df['NAME'].apply(clean_fin_org_names)
sec_df['std_name'] = sec_df['Name'].apply(clean_fin_org_names)
cik_df['std_name'] = cik_df['company_name'].apply(clean_fin_org_names)

In [221]:
print(cik_df.shape)
cik_df.head(10)

(870051, 4)


,Unnamed: 0,company_name,cik,std_name
0,0,!J INC,1438823.0,j
1,1,"#1 A LIFESAFER HOLDINGS, INC.",1509607.0,1 a lifesafer holdings
2,2,#1 ARIZONA DISCOUNT PROPERTIES LLC,1457512.0,1 arizona discount properties
3,3,#1 PAINTBALL CORP,1433777.0,1 paintball
4,4,$ LLC,1427189.0,
5,5,"$AVY, INC.",1655250.0,avy
6,6,& S MEDIA GROUP LLC,1447162.0,s media group
7,7,&TV COMMUNICATIONS INC.,1479357.0,tv communications
8,8,"&VEST DOMESTIC FUND II KPIV, L.P.",1802417.0,vest domestic fund ii kpiv
9,9,&VEST DOMESTIC FUND II LP,1800903.0,vest domestic fund ii lp


In [222]:
duplicates_cik_id = cik_df['cik'].duplicated(keep = False)
# duplicates_cik_id.sum()

duplicates_cik = cik_df[duplicates_cik_id]
duplicates_cik
duplicates_cik[duplicates_cik['cik'] == 1137095.0]

,Unnamed: 0,company_name,cik,std_name
209,209,1 800 MUTUALS ADVISOR SERIES,1137095.0,1 800 mutuals advisor series
210,210,1 800 MUTUALS ADVISORS SERIES,1137095.0,1 800 mutuals advisors series
543341,543341,MUTUALS COM,1137095.0,mutuals com
805992,805992,USA MUTUALS,1137095.0,usa mutuals


In [223]:
print(sec_df.shape)
sec_df.head(10)

(13737, 10)


,index,CIK,Ticker,Name,Exchange,SIC,Business,Incorporated,IRS,std_name
0,0,1090872,A,Agilent Technologies Inc,NYSE,3825.0,CA,DE,770518772.0,agilent technologies
1,1,4281,AA,Alcoa Inc,NYSE,3350.0,PA,PA,250317820.0,alcoa
2,2,1332552,AAACU,Asia Automotive Acquisition Corp,NaN,6770.0,DE,DE,203022522.0,asia automotive acquisition
3,3,1287145,AABB,Asia Broadband Inc,OTC,8200.0,GA,NV,721569126.0,asia broadband
4,4,1024015,AABC,Access Anytime Bancorp Inc,NaN,6035.0,NM,DE,850444597.0,access anytime bancorp
5,5,1099290,AAC,Sinocoking Coal & Coke Chemical Industries Inc,NASDAQ,3312.0,F4,FL,593404233.0,sinocoking coal coke chemical industries
6,6,1264707,AACC,Asset Acceptance Capital Corp,NaN,6153.0,MI,NaN,800076779.0,asset acceptance capital
7,7,849116,AACE,Ace Cash Express Inc,NaN,6099.0,TX,TX,752142963.0,ace cash express
8,8,1409430,AAGC,All American Gold Corp,OTC,1040.0,IN,WY,260665571.0,all american gold
9,9,948846,AAI,Airtran Holdings Inc,NaN,4512.0,FL,NV,582189551.0,airtran holdings


In [224]:
print(compustat_df.shape)
compustat_df.head(100)

(19581, 13)


,Unnamed: 0,gvkey,conm,tic,cusip,cik,sic,naics,gsubind,gind,year1,year2,std_name
0,1,1004,AAR CORP,AIR,000361105,1750.0,5080.0,423860.0,20101010.0,201010.0,1965,2020,aar
1,2,1013,ADC TELECOMMUNICATIONS INC,ADCT.1,000886309,61478.0,3661.0,334210.0,45201020.0,452010.0,1974,2010,adc telecommunications
2,3,1045,AMERICAN AIRLINES GROUP INC,AAL,02376R102,6201.0,4512.0,481111.0,20302010.0,203020.0,1950,2021,american airlines group
3,4,1050,CECO ENVIRONMENTAL CORP,CECE,125141101,3197.0,3564.0,333413.0,20201050.0,202010.0,1974,2021,ceco environmental
4,5,1062,ASA GOLD AND PRECIOUS METALS,ASA,G3156P103,1230869.0,6799.0,523999.0,40203010.0,402030.0,1966,2021,asa gold precious metals
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,1690,APPLE INC,AAPL,037833100,320193.0,3663.0,334220.0,45202030.0,452020.0,1980,2021,apple
96,97,1704,APPLIED MATERIALS INC,AMAT,038222105,6951.0,3559.0,333242.0,45301010.0,453010.0,1972,2021,applied materials
97,98,1706,ENERPAC TOOL GROUP CORP,EPAC,292765104,6955.0,3533.0,333132.0,20106020.0,201060.0,1974,2021,enerpac tool group
98,99,1712,TRECORA RESOURCES,TREC,894648104,7039.0,2911.0,324110.0,15101010.0,151010.0,1976,2021,trecora resources


In [225]:
# cusip is unique 
duplicates_compustat_gvkey = compustat_df['gvkey'].duplicated(keep = False)
# duplicates_compustat_gvkey.sum()

# These are just N/A values, so no need to group by cusip before. 
duplicates_compustat = compustat_df[duplicates_compustat_gvkey]
duplicates_compustat
# All entities are unique in compustat

,Unnamed: 0,gvkey,conm,tic,cusip,cik,sic,naics,gsubind,gind,year1,year2,std_name


In [226]:
print(fdic_df.shape)
fdic_df.head(10)

(25670, 16)


,NAME,NAMEHCR,STALP,STNAME,BKCLASS,ASSET,CERT,FED_RSSD,org_name,commented,Commented,mean_ASSET,median_ASSET,mean_ASSET_type,median_ASSET_type,std_name
0,The Southington Bank and Trust Company,NaN,CT,Connecticut,NM,4.857000e+07,4,573401,the southington bank and trust company,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,southington bank trust
1,Colonial Bank of Waterbury,NaN,CT,Connecticut,NM,6.246550e+08,6,148304,colonial bank of waterbury,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,colonial bank waterbury
2,Fleet Bank of Maine,NaN,ME,Maine,SM,1.699404e+09,8,422406,fleet bank of maine,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,fleet bank maine
3,Union Trust Company,NaN,ME,Maine,SM,5.391690e+08,9,563907,union trust company,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,union trust
4,Northeast Bank of Sanford,NaN,ME,Maine,SM,5.569200e+07,10,112109,northeast bank of sanford,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,northeast bank sanford
5,State Street Bank and Trust Company,State Street Corporation,MA,Massachusetts,SM,2.335429e+11,14,35301,state street bank and trust company,True,Commented,2.565979e+09,145717500,3.941421e+09,218865500,street bank trust
6,BayBank Harvard Trust Company,NaN,MA,Massachusetts,NM,1.435310e+09,18,852209,baybank harvard trust company,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,baybank harvard trust
7,Durfee Attleboro Bank,NaN,MA,Massachusetts,NM,3.468080e+08,20,202402,durfee attleboro bank,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,durfee attleboro bank
8,Bank of New England - North Shore,NaN,MA,Massachusetts,SM,1.532740e+08,21,658205,bank of new england - north shore,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,bank new england - north shore
9,Shawmut Bank of Franklin County,NaN,MA,Massachusetts,NM,1.597270e+08,22,661205,shawmut bank of franklin county,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,shawmut bank franklin


In [227]:
# FED_RSSD is unique, and there are duplicates 
duplicates_FDIC_RSSD = fdic_df['FED_RSSD'].duplicated(keep = False)

duplicates_fdic = fdic_df[duplicates_FDIC_RSSD]
duplicates_fdic

,NAME,NAMEHCR,STALP,STNAME,BKCLASS,ASSET,CERT,FED_RSSD,org_name,commented,Commented,mean_ASSET,median_ASSET,mean_ASSET_type,median_ASSET_type,std_name
1,Colonial Bank of Waterbury,NaN,CT,Connecticut,NM,6.246550e+08,6,148304,colonial bank of waterbury,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,colonial bank waterbury
17,"The Exchange Bank, Attalla, Alabama",NaN,AL,Alabama,NM,6.392700e+07,38,672537,"the exchange bank, attalla, alabama",False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,exchange bank
32,The Exchange Bank of Alabama,"Gadsden Corporation, The",AL,Alabama,NM,2.853190e+08,54,672537,the exchange bank of alabama,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,exchange bank alabama
86,Bank of the Ozarks,Bank Of The Ozarks Inc,AR,Arkansas,NM,1.913922e+10,110,107244,bank of the ozarks,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,bank ozarks
92,"NationsBank of Florida, National Association",NaN,GA,Georgia,N,5.162300e+07,127,147231,"nationsbank of florida, national association",True,Commented,2.565979e+09,145717500,8.836714e+09,126448000,nationsbank florida
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25494,"IndyMac Federal Bank, FSB",NaN,CA,California,SA,2.347791e+10,58912,595373,"indymac federal bank, fsb",False,Did not comment,9.155463e+08,95742500,8.496581e+08,128373500,indymac bank
25512,"Superior Bank, National Association",NaN,FL,Florida,N,2.534417e+09,59026,4262534,"superior bank, national association",False,Did not comment,9.155463e+08,95742500,1.913681e+09,95400500,superior bank
25544,East Boston Savings Bank,NaN,MA,Massachusetts,SB,3.162050e+08,90155,1000100,east boston savings bank,False,Did not comment,9.155463e+08,95742500,8.787780e+08,286179000,east boston savings bank
25616,UniBank for Savings,Ufs Bancorp,MA,Massachusetts,SB,1.619117e+09,90290,709602,unibank for savings,False,Did not comment,9.155463e+08,95742500,8.787780e+08,286179000,unibank for savings


In [228]:
# Test if CIK is already in SEC
sec_ciks = set(sec_df['CIK'].dropna())
cik_ciks = set(cik_df['cik'].dropna())

print(f"Unique CIKs in sec_df: {len(sec_ciks)}")
print(f"Unique CIKs in cik_df: {len(cik_ciks)}")
print(f"Is SEC_Institutions.csv a subset of CIK.csv? {sec_ciks.issubset(cik_ciks)}")

compustat_ciks = set(compustat_df['cik'].dropna())
print(f"CIKs in compustat_df: {len(compustat_ciks)}")
print(f"Is CompustatNames.csv a subset of CIK.csv? {compustat_ciks.issubset(cik_ciks)}")


Unique CIKs in sec_df: 13737
Unique CIKs in cik_df: 806225
Is SEC_Institutions.csv a subset of CIK.csv? True
CIKs in compustat_df: 12835
Is CompustatNames.csv a subset of CIK.csv? False


SEC_Institutions.csv is a subset of CIK.csv --- > Don't need to merge SEC into the crosswalk. 

In [229]:
# This is the final dataframe crosswalk where we will be merging entities into. 

cols = ['standardized_names', 'aliases', 'cik', 'FED_RSSD', 'sources', 'matching_type', 'fuzzy_matching_score']
final_crosswalk_df = pd.DataFrame(columns = cols)

In [230]:
# Merging entities in cik_df based on the unique cik id value. 

grouped_by_cik_id = cik_df.groupby('cik')
confident_matches = []
# count = 0
for cik_value, group in grouped_by_cik_id:
    # count += 1
    # if count % 1000 == 0:
    #   print(count)
    
    if len(group) > 1:
        # Aggregate the data based on cik
        new_match_keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            # Now aggregate the std_name to see all variations found for cik
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['company_name'].dropna().unique()),
            'sources': 'cik',
            'matching_type': 'cik_id_match'
        }
        confident_matches.append(new_match_keys)
    else: 
        unmatched_keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            'standardized_names': group['std_name'].iloc[0],
            'aliases': group['company_name'].iloc[0],
            'sources': 'cik'
        }
        confident_matches.append(unmatched_keys)
         
pd.set_option('display.max_colwidth', None)
enriched_cik_df = pd.DataFrame(confident_matches)
print(f"cik_df reduced to {len(enriched_cik_df)} entities after merging.")

enriched_cik_df

cik_df reduced to 806225 entities after merging.


,cik,standardized_names,aliases,sources,matching_type
0,[3.0],defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,cik,NaN
1,[13.0],corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,cik,NaN
2,[14.0],defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,cik,NaN
3,[17.0],defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,cik,NaN
4,[18.0],nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,cik,NaN
...,...,...,...,...,...
806220,[1937539.0],tipsy lady,"TIPSY LADY, INC.",cik,NaN
806221,[1937546.0],brett anton,BRETT ANTON,cik,NaN
806222,[1937565.0],cw alpha mill apartments,"CW ALPHA MILL APARTMENTS, LP",cik,NaN
806223,[1937584.0],fedder judith ann,FEDDER JUDITH ANN,cik,NaN


In [231]:
# Merge the enriched_cik_df into final_crosswalk_df

final_crosswalk_df = pd.concat([final_crosswalk_df, enriched_cik_df])
final_crosswalk_df

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,[3.0],NaN,cik,NaN,NaN
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,[13.0],NaN,cik,NaN,NaN
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,[14.0],NaN,cik,NaN,NaN
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,[17.0],NaN,cik,NaN,NaN
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,[18.0],NaN,cik,NaN,NaN
...,...,...,...,...,...,...,...
806220,tipsy lady,"TIPSY LADY, INC.",[1937539.0],NaN,cik,NaN,NaN
806221,brett anton,BRETT ANTON,[1937546.0],NaN,cik,NaN,NaN
806222,cw alpha mill apartments,"CW ALPHA MILL APARTMENTS, LP",[1937565.0],NaN,cik,NaN,NaN
806223,fedder judith ann,FEDDER JUDITH ANN,[1937584.0],NaN,cik,NaN,NaN


In [232]:
final_crosswalk_df[final_crosswalk_df['matching_type'] == 'cik_id_match']

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
13,abel noser bd|abel noser,ABEL NOSER CORP /BD|ABEL/NOSER CORP.,[1841.0],NaN,cik,cik_id_match,NaN
15,aberdeen idaho mining|motivnation,"ABERDEEN IDAHO MINING CO|MOTIVNATION, INC.",[1853.0],NaN,cik,cik_id_match,NaN
16,thomson richard william bd|thomson,"THOMSON RICHARD WILLIAM /BD|THOMSON, RICHARD WILLIAM",[1860.0],NaN,cik,cik_id_match,NaN
17,abraham bd|abraham|abraham securities corporation,"ABRAHAM & CO INC /BD|ABRAHAM & CO., INC.|ABRAHAM SECURITIES CORPORATION",[1904.0],NaN,cik,cik_id_match,NaN
18,abrams|homeland securities financial services group|merchanthouse securities|wizer financial,"ABRAMS, ALLAN EDWARD|HOMELAND SECURITIES FINANCIAL SERVICES GROUP, INC.|THE MERCHANTHOUSE SECURITIES, INC.|WIZER FINANCIAL. INC.",[1918.0],NaN,cik,cik_id_match,NaN
...,...,...,...,...,...,...,...
801976,pl fund i|pl fund ii,"PL FUND I, A SERIES OF BETTER CAPITAL, LP|PL FUND II, A SERIES OF BETTER CAPITAL, LP",[1929972.0],NaN,cik,cik_id_match,NaN
802494,metatheory,"METATHEORY, A SERIES OF GCR CJ MASTER LLC|METATHEORY, A SERIES OF GLOBAL COIN VENTURES LLC",[1930747.0],NaN,cik,cik_id_match,NaN
803411,silber mark m|silber moshe mark,SILBER MARK M|SILBER MOSHE MARK,[1932197.0],NaN,cik,cik_id_match,NaN
804306,medlive technology,"MEDLIVE TECHNOLOGY CO., LTD./ADR|MEDLIVE TECHNOLOGY CO., LTD.",[1933644.0],NaN,cik,cik_id_match,NaN


In [233]:
# Test if there are cik ids from compustat_df that match final_crosswalk_df
computstat_ciks = set(compustat_df['cik'].dropna())
cik_ciks = set(cik_df['cik'].dropna())

print(f"compustat_df does not have any cik elements in common with final_crosswalk_df: {cik_ciks.isdisjoint(compustat_ciks)}")

compustat_df does not have any cik elements in common with final_crosswalk_df: False


In [234]:
temp_compustat_df = compustat_df[compustat_df['cik'].notna()].copy()

# This function is used to clean the cik values into a string,
# so that the merge function can be properly called. 
def clean_cik(value):
    # If it's a list, take the first element
    if isinstance(value, list):
        if len(value) > 0:
            value = value[0]
        else:
            return ""
    
    # If it's null, return an empty string
    if pd.isna(value) or value == "":
        return ""
    
    # Convert the value to float, then int, then string to remove the '.0'
    try:
        return str(int(float(value)))
    except (ValueError, TypeError):
        return str(value)

# Apply the cleaning function to both dataframes
temp_compustat_df['cik'] = temp_compustat_df['cik'].astype('string')
temp_compustat_df.loc[:, 'cik'] = temp_compustat_df['cik'].apply(clean_cik)
final_crosswalk_df.loc[:, 'cik'] = final_crosswalk_df['cik'].apply(clean_cik)
temp_compustat_df

,Unnamed: 0,gvkey,conm,tic,cusip,cik,sic,naics,gsubind,gind,year1,year2,std_name
0,1,1004,AAR CORP,AIR,000361105,1750,5080.0,423860.0,20101010.0,201010.0,1965,2020,aar
1,2,1013,ADC TELECOMMUNICATIONS INC,ADCT.1,000886309,61478,3661.0,334210.0,45201020.0,452010.0,1974,2010,adc telecommunications
2,3,1045,AMERICAN AIRLINES GROUP INC,AAL,02376R102,6201,4512.0,481111.0,20302010.0,203020.0,1950,2021,american airlines group
3,4,1050,CECO ENVIRONMENTAL CORP,CECE,125141101,3197,3564.0,333413.0,20201050.0,202010.0,1974,2021,ceco environmental
4,5,1062,ASA GOLD AND PRECIOUS METALS,ASA,G3156P103,1230869,6799.0,523999.0,40203010.0,402030.0,1966,2021,asa gold precious metals
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19575,19576,345980,CONTEXTLOGIC INC,WISH,21077C107,1822250,5961.0,454110.0,25502020.0,255020.0,2015,2021,contextlogic
19576,19577,347007,IMMUNITYBIO INC,IBRX,45256X103,1326110,2836.0,325414.0,35201010.0,352010.0,2019,2021,immunitybio
19577,19578,347085,KAROOOOO LTD,KARO,Y4600W108,1828102,7370.0,518210.0,45103010.0,451030.0,2018,2021,karooooo
19578,19579,349530,NEXTPLAY TECHNOLOGIES INC,NXTP,65344G102,1372183,7310.0,541810.0,50201010.0,502010.0,2021,2021,nextplay technologies


In [235]:
# # Merge compustat_df into final_crosswalk_df based on cik id. 
# Create a dataframe of entities that were not merged called remaining_compustat_df 

final_crosswalk_df = final_crosswalk_df.merge(temp_compustat_df, on = 'cik', how = 'left', suffixes=('','_other'), indicator=True)
final_crosswalk_df

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score,Unnamed: 0,gvkey,conm,tic,cusip,sic,naics,gsubind,gind,year1,year2,std_name,_merge
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
806220,tipsy lady,"TIPSY LADY, INC.",1937539,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
806221,brett anton,BRETT ANTON,1937546,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
806222,cw alpha mill apartments,"CW ALPHA MILL APARTMENTS, LP",1937565,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
806223,fedder judith ann,FEDDER JUDITH ANN,1937584,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [236]:
final_crosswalk_df[final_crosswalk_df['_merge'] == 'both']

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score,Unnamed: 0,gvkey,conm,tic,cusip,sic,naics,gsubind,gind,year1,year2,std_name,_merge
9,aar,AAR CORP,1750,NaN,cik,NaN,NaN,1.0,1004.0,AAR CORP,AIR,000361105,5080.0,423860.0,20101010.0,201010.0,1965.0,2020.0,aar,both
11,abbott laboratories,ABBOTT LABORATORIES,1800,NaN,cik,NaN,NaN,9.0,1078.0,ABBOTT LABORATORIES,ABT,002824100,3845.0,334510.0,35101010.0,351010.0,1950.0,2021.0,abbott laboratories,both
19,abrams industries|servidyne,"ABRAMS INDUSTRIES INC|SERVIDYNE, INC.",1923,NaN,cik,cik_id_match,NaN,10.0,1082.0,SERVIDYNE INC,SERV.1,81765M106,8700.0,541310.0,20201050.0,202010.0,1977.0,2010.0,servidyne,both
23,academic computer systems|worlds com|worlds,"ACADEMIC COMPUTER SYSTEMS INC|WORLDS COM INC|WORLDS INC|WORLDS.COM, INC.",1961,NaN,cik,cik_id_match,NaN,11.0,1084.0,WORLDS INC,WDDD,98159J200,7370.0,519130.0,50203010.0,502030.0,1974.0,2021.0,worlds,both
28,aceto,ACETO CORP,2034,NaN,cik,NaN,NaN,12.0,1094.0,ACETO CORP,ACETQ,004446100,5160.0,424690.0,35102010.0,351020.0,1963.0,2018.0,aceto,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
792113,american riviera bancorp,AMERICAN RIVIERA BANCORP,1916425,NaN,cik,NaN,NaN,15659.0,175449.0,AMERICAN RIVIERA BANCORP,ARBV,02933K103,6020.0,522110.0,40101015.0,401010.0,2012.0,2021.0,american riviera bancorp,both
793609,pbco financial,PBCO FINANCIAL CORP,1918379,NaN,cik,NaN,NaN,12292.0,111872.0,PBCO FINANCIAL CORPORATION,PBCO,69317J101,6020.0,522110.0,40101015.0,401010.0,2014.0,2021.0,pbco financial corporation,both
795593,tudor gold,TUDOR GOLD CORP.,1921050,NaN,cik,NaN,NaN,17898.0,185215.0,TUDOR GOLD CORP,TDRRF,89901P107,1000.0,212.0,15104020.0,151040.0,2014.0,2020.0,tudor gold,both
798648,am resources,AM RESOURCES CORP.,1925124,NaN,cik,NaN,NaN,7609.0,33344.0,AM RESOURES CORP,AMRCF,00179A102,1220.0,212111.0,10102050.0,101020.0,2016.0,2020.0,am resoures,both


In [237]:
# Clean up final_crosswalk_df

mask = final_crosswalk_df['_merge'] == 'both'

# add new std_name
standardized_names = final_crosswalk_df['standardized_names'].astype('string')
new_standardized_name = final_crosswalk_df['std_name'].astype('string')

final_crosswalk_df['standardized_names'] = standardized_names.where(
    new_standardized_name.isna() | (standardized_names == new_standardized_name),
    standardized_names + '|' + new_standardized_name
).fillna(new_standardized_name)

# add new alias
aliases = final_crosswalk_df['aliases'].astype('string')
new_alias = final_crosswalk_df['conm'].astype('string')

final_crosswalk_df['aliases'] = aliases.where(
    new_alias.isna() | (aliases == new_alias),
    aliases + '|' + new_alias
).fillna(new_alias)

# add the source compustat
final_crosswalk_df.loc[mask, 'sources'] = (
    final_crosswalk_df.loc[mask, 'sources']
    .fillna('')
    .where(
        final_crosswalk_df.loc[mask, 'sources'].isna(),
        final_crosswalk_df.loc[mask, 'sources'] + ','
    )
    + 'compustat'
)

# add the entity is matched by cik_id_match
final_crosswalk_df.loc[
    mask,
    'matching_type'
] = 'cik_id_match'

final_crosswalk_df

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score,Unnamed: 0,gvkey,conm,tic,cusip,sic,naics,gsubind,gind,year1,year2,std_name,_merge
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
806220,tipsy lady,"TIPSY LADY, INC.",1937539,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
806221,brett anton,BRETT ANTON,1937546,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
806222,cw alpha mill apartments,"CW ALPHA MILL APARTMENTS, LP",1937565,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
806223,fedder judith ann,FEDDER JUDITH ANN,1937584,NaN,cik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [238]:
print(final_crosswalk_df['_merge'].value_counts())

_merge
left_only     793409
both           12816
right_only         0
Name: count, dtype: int64


In [239]:
# Cleaning up final_crosswalk_df

final_crosswalk_df = final_crosswalk_df.drop(columns=['Unnamed: 0','gvkey', 'conm', 'tic', 'cusip', 'sic', \
                                                      'naics', 'gsubind', 'gind', 'year1', 'year2', 'std_name', '_merge'])

In [240]:
final_crosswalk_df

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN
...,...,...,...,...,...,...,...
806220,tipsy lady,"TIPSY LADY, INC.",1937539,NaN,cik,NaN,NaN
806221,brett anton,BRETT ANTON,1937546,NaN,cik,NaN,NaN
806222,cw alpha mill apartments,"CW ALPHA MILL APARTMENTS, LP",1937565,NaN,cik,NaN,NaN
806223,fedder judith ann,FEDDER JUDITH ANN,1937584,NaN,cik,NaN,NaN


In [241]:
rejected_compustat_df = temp_compustat_df[~temp_compustat_df['cik'].isin(final_crosswalk_df['cik'])].reset_index(drop=True)
rejected_compustat_df


,Unnamed: 0,gvkey,conm,tic,cusip,cik,sic,naics,gsubind,gind,year1,year2,std_name
0,657,6000,INTECH INC,INTE.,458095106,110640,3674.0,334413.0,45205020.0,452050.0,1981,2010,intech
1,724,6423,KEYSTONE CAMERA PRODUCTS,KYC.,493397103,6125,3861.0,325992.0,25202020.0,252020.0,1971,2011,keystone camera products
2,965,8145,ON-LINE SOFTWARE INTL,OSI.3,682180104,705406,7372.0,511210.0,45103010.0,451030.0,1981,2010,-line software intl
3,1348,11137,VERIT INDUSTRIES,7621B,923434203,103274,1531.0,233210.0,25201030.0,252010.0,1968,2011,verit industries
4,1448,11878,CAFES ONE -LP,6870B,127698108,805275,6794.0,533110.0,20201030.0,202010.0,1985,2011,cafes one -lp
5,1506,12161,AUTODIE CORP,3ADIE,052770104,778705,3540.0,3335.0,20106020.0,201060.0,1983,2011,autodie
6,1533,12324,CLIFF ENGLE LTD,3CLIFE,186901203,793596,2253.0,315191.0,25203010.0,252030.0,1983,2010,cliff engle
7,1639,12860,LAURION MINERALS EXPL INC,LMEFF,519322101,751146,1040.0,212221.0,15104030.0,151040.0,1984,2021,laurion minerals expl
8,1823,13958,GOLDOME BUFFALO NY,GDM.3,380934109,858448,6035.0,522120.0,40101010.0,401010.0,1985,2011,goldome buffalo ny
9,1893,14265,TUDOR CORP LTD,TDRLF,898901103,824726,1311.0,211111.0,10102020.0,101020.0,1984,2015,tudor


In [242]:
print(f"The number of compustat entities that didn't match anything: {len(rejected_compustat_df)}")

The number of compustat entities that didn't match anything: 19


In [243]:
rejected_compustat_df = rejected_compustat_df.drop(columns = ['Unnamed: 0', 'gvkey', 'tic', 'cusip', 'sic', 
                                                              'naics', 'gsubind', 'gind', 'year1', 'year2', ])
rejected_compustat_df = rejected_compustat_df.rename(columns={'std_name': 'standardized_names', 'conm': 'aliases'})
rejected_compustat_df['sources'] = 'compustat'

In [244]:
rejected_compustat_df

,aliases,cik,standardized_names,sources
0,INTECH INC,110640,intech,compustat
1,KEYSTONE CAMERA PRODUCTS,6125,keystone camera products,compustat
2,ON-LINE SOFTWARE INTL,705406,-line software intl,compustat
3,VERIT INDUSTRIES,103274,verit industries,compustat
4,CAFES ONE -LP,805275,cafes one -lp,compustat
5,AUTODIE CORP,778705,autodie,compustat
6,CLIFF ENGLE LTD,793596,cliff engle,compustat
7,LAURION MINERALS EXPL INC,751146,laurion minerals expl,compustat
8,GOLDOME BUFFALO NY,858448,goldome buffalo ny,compustat
9,TUDOR CORP LTD,824726,tudor,compustat


In [245]:
final_crosswalk_df = pd.concat([final_crosswalk_df, rejected_compustat_df], axis = 0, ignore_index=True)

In [246]:
final_crosswalk_df

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN
...,...,...,...,...,...,...,...
806239,country financial cp,TOWN & COUNTRY FINANCIAL CP,98840,NaN,compustat,NaN,NaN
806240,polestar exploration,POLESTAR EXPLORATION INC,795261,NaN,compustat,NaN,NaN
806241,kiplin metals,KIPLIN METALS INC,930537,NaN,compustat,NaN,NaN
806242,etrion,ETRION CORP,1066690,NaN,compustat,NaN,NaN


In [247]:
### THIS IS A INEFFICIENT ALGORITHM TO MATCH COMPUSTAT INTO FINAL_CROSSWALK_DF BASED ON CIK

# Merge compustat_df into final_crosswalk_df based on cik id. 
# Create a dataframe of entities that were not merged called remaining_compustat_df 

# New df with the whole row of the rows of compustat with a cik id

# TO DO: Fix to improve time complexity, use merge function of pandas

# temp_compustat_df = compustat_df['cik'].dropna()
# temp_compustat_df = compustat_df.loc[temp_compustat_df.index]
# count = 0 
# for i in range(len(temp_compustat_df)):
#     count += 1
#     if count % 1000 == 0:
#       print(count)
#     for j in range(len(final_crosswalk_df)):
#         current_compustat_row = temp_compustat_df.iloc[i]
#         current_crosswalk_row = final_crosswalk_df.iloc[j]
#         if current_compustat_row['cik'] == current_crosswalk_row['cik']:
#             current_standardized_name = final_crosswalk_df.iloc[j]['standardized_names']
#             final_crosswalk_df.iloc[j, 0] = current_standardized_name + "," + current_compustat_row['std_name']
            
#             current_alias_name = final_crosswalk_df.iloc[j]['aliases']
#             final_crosswalk_df.iloc[j, 1] = current_alias_name + "," + current_compustat_row['conm']

#             current_sources = final_crosswalk_df.iloc[j]['sources']
#             final_crosswalk_df.iloc[j, 4] = current_sources + "," + "compustat"
            
#             current_matching_types = final_crosswalk_df.iloc[j]['matching_type']
#             if isinstance(current_matching_types, float):
#                 # If it's a float (NaN), treat it as an empty string for the check
#                 current_matching_types = ""
#             if 'cik_id' in current_matching_types:
#                 continue
#             final_crosswalk_df.iloc[j, 5] = current_matching_types + "," + "cik_id"        
            

In [248]:
fdic_df.head()

,NAME,NAMEHCR,STALP,STNAME,BKCLASS,ASSET,CERT,FED_RSSD,org_name,commented,Commented,mean_ASSET,median_ASSET,mean_ASSET_type,median_ASSET_type,std_name
0,The Southington Bank and Trust Company,NaN,CT,Connecticut,NM,4.857000e+07,4,573401,the southington bank and trust company,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,southington bank trust
1,Colonial Bank of Waterbury,NaN,CT,Connecticut,NM,6.246550e+08,6,148304,colonial bank of waterbury,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000,colonial bank waterbury
2,Fleet Bank of Maine,NaN,ME,Maine,SM,1.699404e+09,8,422406,fleet bank of maine,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,fleet bank maine
3,Union Trust Company,NaN,ME,Maine,SM,5.391690e+08,9,563907,union trust company,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,union trust
4,Northeast Bank of Sanford,NaN,ME,Maine,SM,5.569200e+07,10,112109,northeast bank of sanford,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000,northeast bank sanford


In [249]:
# Merge entities in fdic based on FED_RSSD values with itself to remove duplicates. 
# New df called enriched_fdic_df

grouped_by_FED_RSSD = fdic_df.groupby('FED_RSSD')
confident_matches = []
# count = 0
for FED_RSSD_value, group in grouped_by_FED_RSSD:
    # count += 1
    # if count % 1000 == 0:
    #   print(count)
    
    if len(group) > 1:
        # Aggregate the data based on cik
        new_match_keys = {
            'FED_RSSD': group['FED_RSSD'].dropna().unique().tolist(),
            # Now aggregate the std_name to see all variations found for FED_RSSD
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['NAME'].dropna().unique()),
            'sources': 'fdic',
            'matching_type': 'FED_RSSD_match'
        }
        confident_matches.append(new_match_keys)
    else: 
        unmatched_keys = {
            'FED_RSSD': group['FED_RSSD'].dropna().unique().tolist(),
            'standardized_names': group['std_name'].iloc[0],
            'aliases': group['NAME'].iloc[0],
            'sources': 'fdic'
        }
        confident_matches.append(unmatched_keys)
         
pd.set_option('display.max_colwidth', None)
enriched_fdic_df = pd.DataFrame(confident_matches)
print(f"fdic_df reduced to {len(enriched_cik_df)} entities after merging.")

enriched_fdic_df

fdic_df reduced to 806225 entities after merging.


,FED_RSSD,standardized_names,aliases,sources,matching_type
0,[0],american savings loan|mckinley savings loan|united first savings loan|cheltenham savings loan|liberty savings|home bank florida|pine belt savings loan|concho valley savings loan|colonial savings loan|first savings bank|sterling savings|barbary coast savings bank|sequoia savings bank|citizens savings loan|truman savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
1,[28],1st atlantic bank,1st Atlantic Bank,fdic,NaN
2,[37],bank hancock,Bank of Hancock County,fdic,NaN
3,[46],republic bank,Republic Bank,fdic,NaN
4,[55],bank littlefork,State Bank of Littlefork,fdic,NaN
...,...,...,...,...,...
24716,[4536084],bank bird--hand,Bank of Bird-in-Hand,fdic,NaN
24717,[4569167],new traditions bank,New Traditions Bank,fdic,NaN
24718,[4845861],primary bank,Primary Bank,fdic,NaN
24719,[5050028],international bank,International Bank of Commerce,fdic,NaN


In [250]:
enriched_fdic_df[enriched_fdic_df['matching_type'] == 'FED_RSSD_match']

,FED_RSSD,standardized_names,aliases,sources,matching_type
0,[0],american savings loan|mckinley savings loan|united first savings loan|cheltenham savings loan|liberty savings|home bank florida|pine belt savings loan|concho valley savings loan|colonial savings loan|first savings bank|sterling savings|barbary coast savings bank|sequoia savings bank|citizens savings loan|truman savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
42,[1874],home unity savings loan,Home Unity Savings and Loan Association|Home Unity Federal Savings and Loan Association,fdic,FED_RSSD_match
55,[2509],connecticut bank trust,"The Connecticut Bank and Trust Company, National Association|The Connecticut Bank and Trust Company",fdic,FED_RSSD_match
78,[3878],connecticut savings loan,Connecticut Savings and Loan Association|Connecticut Federal Savings and Loan Association,fdic,FED_RSSD_match
92,[4455],first bank,First State Bank,fdic,FED_RSSD_match
...,...,...,...,...,...
23546,[2949710],national bank|triumph community bank,"THE National Bank|Triumph Community Bank, National Association",fdic,FED_RSSD_match
23853,[3195037],bank escondido|sunrise bank,Bank of Escondido|Sunrise Bank,fdic,FED_RSSD_match
24445,[3555695],atlantic capital bank,"Atlantic Capital Bank, National Association|Atlantic Capital Bank",fdic,FED_RSSD_match
24521,[3607062],lone star bank west texas,Lone Star State Bank of West Texas|Lone Star State Bank of West Texas,fdic,FED_RSSD_match


In [251]:
# Checking to see which entities in fdic are qualified to be matched 
# based on an exact standardized_name match. Standardized names with more than 
# one appearence in enriched fdic do not qualify to be matched into final_crosswalk_df 
# based on this method, because it is ambiguous as to whether which one matches the entity 
# in final_crosswalk_df when we know they are different because they have different 
# FED_RSSD ids. 

# unqualified_for_standardized_names_matching = enriched_fdic_df['standardized_names'].duplicated(keep=False)
# qualified_for_standardized_names_matching = enriched_fdic_df[~unqualified_for_standardized_names_matching]

In [252]:
# qualified_for_standardized_names_matching[qualified_for_standardized_names_matching['matching_type'] == 'FED_RSSD_match']

In [253]:
# Merge enriched_fdic into final_crosswalk_df based on exact standardized_name matching. 

final_crosswalk_df_exploded = (
    final_crosswalk_df.assign(standardized_names = final_crosswalk_df['standardized_names'].str.split('|'))
    .explode('standardized_names')  
)

final_crosswalk_df_exploded['standardized_names'] = (
    final_crosswalk_df_exploded['standardized_names'].str.strip()
)

final_crosswalk_df_exploded

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN
...,...,...,...,...,...,...,...
806239,country financial cp,TOWN & COUNTRY FINANCIAL CP,98840,NaN,compustat,NaN,NaN
806240,polestar exploration,POLESTAR EXPLORATION INC,795261,NaN,compustat,NaN,NaN
806241,kiplin metals,KIPLIN METALS INC,930537,NaN,compustat,NaN,NaN
806242,etrion,ETRION CORP,1066690,NaN,compustat,NaN,NaN


In [254]:
enriched_fdic_df_exploded = (
    enriched_fdic_df.assign(standardized_names = enriched_fdic_df['standardized_names'].str.split('|'))
    .explode('standardized_names')  
)

enriched_fdic_df_exploded['standardized_names'] = (
    enriched_fdic_df_exploded['standardized_names'].str.strip()
)

enriched_fdic_df_exploded

,FED_RSSD,standardized_names,aliases,sources,matching_type
0,[0],american savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
0,[0],mckinley savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
0,[0],united first savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
0,[0],cheltenham savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
0,[0],liberty savings,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
...,...,...,...,...,...
24716,[4536084],bank bird--hand,Bank of Bird-in-Hand,fdic,NaN
24717,[4569167],new traditions bank,New Traditions Bank,fdic,NaN
24718,[4845861],primary bank,Primary Bank,fdic,NaN
24719,[5050028],international bank,International Bank of Commerce,fdic,NaN


In [255]:
unqualified_for_standardized_names_matching_fdic = enriched_fdic_df_exploded['standardized_names'].duplicated(keep=False)
qualified_for_standardized_names_matching__fdic_exploded = enriched_fdic_df_exploded[~unqualified_for_standardized_names_matching_fdic]
qualified_for_standardized_names_matching__fdic_exploded

,FED_RSSD,standardized_names,aliases,sources,matching_type
0,[0],mckinley savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
0,[0],cheltenham savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
0,[0],home bank florida,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
0,[0],pine belt savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
0,[0],concho valley savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
...,...,...,...,...,...
24714,[4262534],cadence bank,"Cadence Bank, N.A.|Superior Bank, National Association",fdic,FED_RSSD_match
24715,[4262543],alostar bank,AloStar Bank of Commerce,fdic,NaN
24716,[4536084],bank bird--hand,Bank of Bird-in-Hand,fdic,NaN
24717,[4569167],new traditions bank,New Traditions Bank,fdic,NaN


In [256]:
unqualified_for_standardized_names_matching_final = final_crosswalk_df_exploded['standardized_names'].duplicated(keep=False)
qualified_for_standardized_names_matching_final_exploded = final_crosswalk_df_exploded[~unqualified_for_standardized_names_matching_final]
qualified_for_standardized_names_matching_final_exploded

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN
...,...,...,...,...,...,...,...
806239,country financial cp,TOWN & COUNTRY FINANCIAL CP,98840,NaN,compustat,NaN,NaN
806240,polestar exploration,POLESTAR EXPLORATION INC,795261,NaN,compustat,NaN,NaN
806241,kiplin metals,KIPLIN METALS INC,930537,NaN,compustat,NaN,NaN
806242,etrion,ETRION CORP,1066690,NaN,compustat,NaN,NaN


In [257]:
qualified_for_standardized_names_matching_final_exploded['exploded_index'] = qualified_for_standardized_names_matching_final_exploded.index
qualified_for_standardized_names_matching__fdic_exploded['exploded_index'] = qualified_for_standardized_names_matching__fdic_exploded.index

/var/folders/9p/px6hv6w54h9f61mj90xkdjxm0000gn/T/ipykernel_39651/785459677.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qualified_for_standardized_names_matching_final_exploded['exploded_index'] = qualified_for_standardized_names_matching_final_exploded.index
/var/folders/9p/px6hv6w54h9f61mj90xkdjxm0000gn/T/ipykernel_39651/785459677.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  qualified_for_standardized_names_matching__fdic_exploded['exploded_index'] = qualified_for_standardized_names_matching

In [258]:
overlap = final_crosswalk_df.columns.intersection(qualified_for_standardized_names_matching__fdic_exploded.columns)

qualified_for_standardized_names_matching__fdic_exploded = qualified_for_standardized_names_matching__fdic_exploded.rename(
    columns={
        c: f'df2_{c}'
        for c in overlap
        if c != 'standardized_names'
    }
)


In [259]:
qualified_for_standardized_names_matching__fdic_exploded[qualified_for_standardized_names_matching__fdic_exploded['df2_aliases'] == 'Bank of Hawaii']

,df2_FED_RSSD,standardized_names,df2_aliases,df2_sources,df2_matching_type,exploded_index
16584,[795968],bank hawaii,Bank of Hawaii,fdic,NaN,16584


In [260]:

qualified_for_standardized_names_matching_final_exploded[qualified_for_standardized_names_matching_final_exploded['standardized_names'] == 'bank hawaii']

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score,exploded_index


In [261]:
merged = qualified_for_standardized_names_matching_final_exploded.merge(
    qualified_for_standardized_names_matching__fdic_exploded,
    on='standardized_names',
    how='left',
    suffixes=('', '_df2'),
    indicator=True
)

In [262]:
merged

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score,exploded_index,df2_FED_RSSD,df2_aliases,df2_sources,df2_matching_type,exploded_index_df2,_merge
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,left_only
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,left_only
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,left_only
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,left_only
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810113,country financial cp,TOWN & COUNTRY FINANCIAL CP,98840,NaN,compustat,NaN,NaN,806239,NaN,NaN,NaN,NaN,NaN,left_only
810114,polestar exploration,POLESTAR EXPLORATION INC,795261,NaN,compustat,NaN,NaN,806240,NaN,NaN,NaN,NaN,NaN,left_only
810115,kiplin metals,KIPLIN METALS INC,930537,NaN,compustat,NaN,NaN,806241,NaN,NaN,NaN,NaN,NaN,left_only
810116,etrion,ETRION CORP,1066690,NaN,compustat,NaN,NaN,806242,NaN,NaN,NaN,NaN,NaN,left_only


In [263]:
merged[merged['_merge'] == 'both']

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score,exploded_index,df2_FED_RSSD,df2_aliases,df2_sources,df2_matching_type,exploded_index_df2,_merge
749,bb t financial,BB&T FINANCIAL CORP|BB&T FUNDS,13839,NaN,cik,cik_id_match,NaN,593,[2689463],"BB&T Financial, FSB",fdic,NaN,22933.0,both
3278,united california bank,BANK OF THE WEST|SANWA BANK CALIFORNIA|UNITED CALIFORNIA BANK,59951,NaN,cik,cik_id_match,NaN,2559,[438368],UNITED CALIFORNIA BANK,fdic,NaN,8974.0,both
3517,mellon bank,MELLON BANK CORP|MELLON FINANCIAL CORP,64782,NaN,cik,cik_id_match,NaN,2751,[825904],"Mellon Bank, F.S.B.",fdic,NaN,17221.0,both
4769,rockland trust,ROCKLAND TRUST CO,84616,NaN,cik,NaN,NaN,3796,[613008],Rockland Trust Company,fdic,NaN,12530.0,both
5593,trust new jersey,TRUST CO OF NEW JERSEY,99982,NaN,cik,NaN,NaN,4462,[31303],The Trust Company of New Jersey,fdic,NaN,658.0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
754201,choice financial group,"CHOICE FINANCIAL GROUP, LLC",1857130,NaN,cik,NaN,NaN,743062,[826956],Choice Financial Group,fdic,NaN,17239.0,both
761051,heartland bank trust,HEARTLAND BANK & TRUST CO,1866040,NaN,cik,NaN,NaN,750651,[426534],Heartland Bank and Trust Company,fdic,NaN,8753.0,both
765521,morgan stanley bank,"MORGAN STANLEY BANK, N.A.",1871769,NaN,cik,NaN,NaN,755554,[1456501],"Morgan Stanley Bank, National Association",fdic,NaN,21917.0,both
791298,midwest heritage bank,"MIDWEST HERITAGE BANK, FSB",1906805,NaN,cik,NaN,NaN,784793,[203043],"Midwest Heritage Bank, FSB",fdic,NaN,4138.0,both


In [264]:
# Clean up merged before turning back into final_crosswalk_df

mask = merged['_merge'] == 'both'

# add new alias
aliases = merged['aliases'].astype('string')
new_alias = merged['df2_aliases'].astype('string')

merged['aliases'] = aliases.where(
    new_alias.isna() | (aliases == new_alias),
    aliases + '|' + new_alias
).fillna(new_alias)

# add the source fdic
merged.loc[mask, 'sources'] = (
    merged.loc[mask, 'sources']
    .fillna('')
    .where(
        merged.loc[mask, 'sources'].isna(),
        merged.loc[mask, 'sources'] + ','
    )
    + 'fdic'
)

merged.loc[mask, 'FED_RSSD'] = (
    merged.loc[mask, 'df2_FED_RSSD']
)

# add the entity is matched by standardized_name_matching
merged.loc[mask, 'matching_type'] = (
    merged.loc[mask, 'matching_type']
    .fillna('')
    .where(
        merged.loc[mask, 'matching_type'].isna(),
        merged.loc[mask, 'matching_type'] + ','
    )
    + merged.loc[mask, 'df2_matching_type'].fillna('') + 'standardized_name_matching'
)

merged

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score,exploded_index,df2_FED_RSSD,df2_aliases,df2_sources,df2_matching_type,exploded_index_df2,_merge
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,left_only
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,left_only
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,left_only
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,left_only
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810113,country financial cp,TOWN & COUNTRY FINANCIAL CP,98840,NaN,compustat,NaN,NaN,806239,NaN,NaN,NaN,NaN,NaN,left_only
810114,polestar exploration,POLESTAR EXPLORATION INC,795261,NaN,compustat,NaN,NaN,806240,NaN,NaN,NaN,NaN,NaN,left_only
810115,kiplin metals,KIPLIN METALS INC,930537,NaN,compustat,NaN,NaN,806241,NaN,NaN,NaN,NaN,NaN,left_only
810116,etrion,ETRION CORP,1066690,NaN,compustat,NaN,NaN,806242,NaN,NaN,NaN,NaN,NaN,left_only


In [265]:
merged[merged['_merge'] == 'both']

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score,exploded_index,df2_FED_RSSD,df2_aliases,df2_sources,df2_matching_type,exploded_index_df2,_merge
749,bb t financial,"BB&T FINANCIAL CORP|BB&T FUNDS|BB&T Financial, FSB",13839,[2689463],"cik,fdic","cik_id_match,standardized_name_matching",NaN,593,[2689463],"BB&T Financial, FSB",fdic,NaN,22933.0,both
3278,united california bank,BANK OF THE WEST|SANWA BANK CALIFORNIA|UNITED CALIFORNIA BANK|UNITED CALIFORNIA BANK,59951,[438368],"cik,fdic","cik_id_match,standardized_name_matching",NaN,2559,[438368],UNITED CALIFORNIA BANK,fdic,NaN,8974.0,both
3517,mellon bank,"MELLON BANK CORP|MELLON FINANCIAL CORP|Mellon Bank, F.S.B.",64782,[825904],"cik,fdic","cik_id_match,standardized_name_matching",NaN,2751,[825904],"Mellon Bank, F.S.B.",fdic,NaN,17221.0,both
4769,rockland trust,ROCKLAND TRUST CO|Rockland Trust Company,84616,[613008],"cik,fdic",standardized_name_matching,NaN,3796,[613008],Rockland Trust Company,fdic,NaN,12530.0,both
5593,trust new jersey,TRUST CO OF NEW JERSEY|The Trust Company of New Jersey,99982,[31303],"cik,fdic",standardized_name_matching,NaN,4462,[31303],The Trust Company of New Jersey,fdic,NaN,658.0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
754201,choice financial group,"CHOICE FINANCIAL GROUP, LLC|Choice Financial Group",1857130,[826956],"cik,fdic",standardized_name_matching,NaN,743062,[826956],Choice Financial Group,fdic,NaN,17239.0,both
761051,heartland bank trust,HEARTLAND BANK & TRUST CO|Heartland Bank and Trust Company,1866040,[426534],"cik,fdic",standardized_name_matching,NaN,750651,[426534],Heartland Bank and Trust Company,fdic,NaN,8753.0,both
765521,morgan stanley bank,"MORGAN STANLEY BANK, N.A.|Morgan Stanley Bank, National Association",1871769,[1456501],"cik,fdic",standardized_name_matching,NaN,755554,[1456501],"Morgan Stanley Bank, National Association",fdic,NaN,21917.0,both
791298,midwest heritage bank,"MIDWEST HERITAGE BANK, FSB|Midwest Heritage Bank, FSB",1906805,[203043],"cik,fdic",standardized_name_matching,NaN,784793,[203043],"Midwest Heritage Bank, FSB",fdic,NaN,4138.0,both


In [266]:
merged = merged.drop(columns=['df2_FED_RSSD','df2_aliases', 'df2_sources', 'df2_matching_type', 'exploded_index', 'exploded_index_df2', '_merge'])

In [267]:
merged

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,NaN,cik,NaN,NaN
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,NaN,cik,NaN,NaN
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,NaN,cik,NaN,NaN
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,NaN,cik,NaN,NaN
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,NaN,cik,NaN,NaN
...,...,...,...,...,...,...,...
810113,country financial cp,TOWN & COUNTRY FINANCIAL CP,98840,NaN,compustat,NaN,NaN
810114,polestar exploration,POLESTAR EXPLORATION INC,795261,NaN,compustat,NaN,NaN
810115,kiplin metals,KIPLIN METALS INC,930537,NaN,compustat,NaN,NaN
810116,etrion,ETRION CORP,1066690,NaN,compustat,NaN,NaN


In [268]:
qualified_for_fuzzy_matching = qualified_for_standardized_names_matching__fdic_exploded[\
    ~qualified_for_standardized_names_matching__fdic_exploded['standardized_names'].isin(merged['standardized_names'])].reset_index(drop = True) 
qualified_for_fuzzy_matching

,df2_FED_RSSD,standardized_names,df2_aliases,df2_sources,df2_matching_type,exploded_index
0,[0],mckinley savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match,0
1,[0],cheltenham savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match,0
2,[0],home bank florida,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match,0
3,[0],pine belt savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match,0
4,[0],concho valley savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match,0
...,...,...,...,...,...,...
15991,[4210227],nbh bank,NBH Bank,fdic,NaN,24713
15992,[4262534],cadence bank,"Cadence Bank, N.A.|Superior Bank, National Association",fdic,FED_RSSD_match,24714
15993,[4536084],bank bird--hand,Bank of Bird-in-Hand,fdic,NaN,24716
15994,[4569167],new traditions bank,New Traditions Bank,fdic,NaN,24717


In [269]:
def normalize_to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [int(i) for i in x]  # convert to int
    return [int(x)]

In [270]:
# Create a testing mode to sample data for faster fuzzy matching
TESTING_MODE = True
SAMPLE_FRAC = 0.01 # choose percent of data to sample in testing mode

if TESTING_MODE:
    print(f"--- RUNNING FUZZY MATCHING IN TESTING MODE (Sample: {SAMPLE_FRAC*100}%) ---")
    
    # Take a random sample of your 'unmatched_for_fuzzy' DataFrame
    qualified_for_fuzzy_matching = qualified_for_fuzzy_matching.sample(frac=SAMPLE_FRAC, random_state=42)
    
    print(f"  'qualified_for_fuzzy_matching' sampled to: {len(qualified_for_fuzzy_matching)} rows")
    print("-------------------------------------------------")
    
else:
    print(f"--- RUNNING FUZZY MATCHING IN FULL PRODUCTION MODE ---")
    print(f"  'qualified_for_fuzzy_matching' full size: {len(qualified_for_fuzzy_matching)} rows")
    print("-------------------------------------------------")

--- RUNNING FUZZY MATCHING IN TESTING MODE (Sample: 1.0%) ---
  'qualified_for_fuzzy_matching' sampled to: 160 rows
-------------------------------------------------


In [271]:
from rapidfuzz import process, fuzz

# Merge qualified_for_fuzzy_matching into merged based on fuzzy matching. 
# 1. Go through each entity in remaining qualified_for_fuzzy_matching and go through the final_crosswalk_df entities
# 2. If the score is higher than the desired threshold, append the values to the existing Series. 

# Prepare list of all names from merged, flattened for multiple names per row
crosswalk_alias_list = []
crosswalk_idx_list = []
merged['FED_RSSD'] = merged['FED_RSSD'].apply(normalize_to_list)
merged['fuzzy_matching_score'] = [[] for _ in range(len(merged))]

print('Beginning string splitting')
count = 0
for idx, row in merged.iterrows():
    count += 1 
    if count % 1000 == 0:
        print(count)
    names = row['aliases'].split('|')
    for name in names:
        crosswalk_alias_list.append(name)
        crosswalk_idx_list.append(idx)

# Now go through qualified_for_fuzzy_matching
count = 0
print('beginning matching')
for i, new_row in qualified_for_fuzzy_matching.iterrows():
    count += 1 
    if count % 50 == 0:
        print(count)
    
    new_alias = new_row['df2_aliases']
    # Find best matches with threshold 90
    matches = process.extract(
        new_alias,
        crosswalk_alias_list,
        scorer=fuzz.token_set_ratio,
        score_cutoff=50
    )
    
    for match_name, score, match_pos in matches:
        idx = crosswalk_idx_list[match_pos]
        
        # Skip merge if 'fdic' already in sources
        existing_sources = merged.at[idx, 'sources']
        if existing_sources and 'fdic' in existing_sources.split(','):
            continue

        # Update merged row
        merged.at[idx, 'aliases'] = (
            merged.at[idx, 'aliases'] + '|' + new_alias
        )
        mask = qualified_for_fuzzy_matching['df2_aliases'] == new_alias
        alias_val = qualified_for_fuzzy_matching.loc[mask, 'standardized_names'].iloc[0]
        merged.at[idx, 'standardized_names'] = (
            merged.at[idx, 'standardized_names'] + '|' + alias_val
        )
        rssd_val = qualified_for_fuzzy_matching.loc[mask, 'df2_FED_RSSD'].iloc[0]
        # If rssd_val is a list, get the first element
        if isinstance(rssd_val, list) and len(rssd_val) > 0:
            rssd_val = rssd_val[0]
        merged.at[idx, 'FED_RSSD'].append(int(rssd_val))

        val_source = merged.at[idx, 'sources']
        merged.at[idx, 'sources'] = (
            ("" if pd.isna(val_source) else val_source + ',') + 'fdic'
        )
        val_matching_type = merged.at[idx, 'matching_type']
        merged.at[idx, 'matching_type'] = (
            ("" if pd.isna(val_matching_type) else val_matching_type + ',') + 'fuzzy_matching'
        )
        # Append fuzzy score to the column
        merged.at[idx, 'fuzzy_matching_score'].append(score)


Beginning string splitting
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000
113000
114000
115000
116000
117000
118000
119000
120000
121000
122000
123000
124000
125000
126000
127000
128000
129000
130000
131000
132000
133000
134000
135000
136000
137000
138000
139000
140000
141000
142000
143000
144000
145000
146000
147000
148000
149000
150000
151000
152000
153000
154000
155

In [272]:
fuzzy_rows = merged[merged['matching_type'].str.contains('fuzzy_matching', na=False)]
fuzzy_rows

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
1485,dempsey bd|e trade bank,"DEMPSEY & CO LLC /BD|DEMPSEY & COMPANY, L.L.C.|E*TRADE CAPITAL MARKETS - EXECUTION SERVICES LLC|E*TRADE Bank",28051,[370271],"cik,fdic","cik_id_match,fuzzy_matching",[73.6842105263158]
1486,dempsey|e trade bank,"DEMPSEY & CO LLC /BD|DEMPSEY & COMPANY, L.L.C.|E*TRADE CAPITAL MARKETS - EXECUTION SERVICES LLC|E*TRADE Bank",28051,[370271],"cik,fdic","cik_id_match,fuzzy_matching",[73.6842105263158]
1487,e trade capital markets - execution services|e trade bank,"DEMPSEY & CO LLC /BD|DEMPSEY & COMPANY, L.L.C.|E*TRADE CAPITAL MARKETS - EXECUTION SERVICES LLC|E*TRADE Bank",28051,[370271],"cik,fdic","cik_id_match,fuzzy_matching",[73.6842105263158]
27655,e trade clearing|e trade bank,"E*TRADE CLEARING LLC|E*TRADE INSTITUTIONAL SECURITIES, INC.|E*TRADE Bank",851591,[370271],"cik,fdic","cik_id_match,fuzzy_matching",[73.6842105263158]
42426,alantec|bank lantec,ALANTEC CORP|Bank @LANTEC,916073,[546384],"cik,fdic",fuzzy_matching,[58.333333333333336]
141633,boone|central bank boone,BOONE COUNTY CORP|CENTRAL BANK OF BOONE COUNTY,1168844,[299046],"cik,fdic",fuzzy_matching,[82.75862068965517]
796298,central bank brazil|central bank boone,CENTRAL BANK OF BRAZIL|CENTRAL BANK OF BOONE COUNTY,1914083,[299046],"cik,fdic",fuzzy_matching,[81.08108108108108]
798989,saudi central bank|central bank boone,SAUDI CENTRAL BANK|CENTRAL BANK OF BOONE COUNTY,1918181,[299046],"cik,fdic",fuzzy_matching,[80.0]


In [273]:
enriched_fdic_df['FED_RSSD'] = (
    enriched_fdic_df['FED_RSSD']
    .str[0]
    .astype('Int64')   
)

In [274]:
merged['FED_RSSD'] = (
    merged['FED_RSSD']
    .str[0]
    .astype('Int64')  
)

In [275]:

remaining_fdic_df = enriched_fdic_df[~enriched_fdic_df['FED_RSSD'].isin(merged['FED_RSSD'])]
final_crosswalk_df = pd.concat([merged, remaining_fdic_df], ignore_index=True)

In [276]:
enriched_fdic_df

,FED_RSSD,standardized_names,aliases,sources,matching_type
0,0,american savings loan|mckinley savings loan|united first savings loan|cheltenham savings loan|liberty savings|home bank florida|pine belt savings loan|concho valley savings loan|colonial savings loan|first savings bank|sterling savings|barbary coast savings bank|sequoia savings bank|citizens savings loan|truman savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
1,28,1st atlantic bank,1st Atlantic Bank,fdic,NaN
2,37,bank hancock,Bank of Hancock County,fdic,NaN
3,46,republic bank,Republic Bank,fdic,NaN
4,55,bank littlefork,State Bank of Littlefork,fdic,NaN
...,...,...,...,...,...
24716,4536084,bank bird--hand,Bank of Bird-in-Hand,fdic,NaN
24717,4569167,new traditions bank,New Traditions Bank,fdic,NaN
24718,4845861,primary bank,Primary Bank,fdic,NaN
24719,5050028,international bank,International Bank of Commerce,fdic,NaN


In [277]:
merged[merged['FED_RSSD'].notna()]

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
749,bb t financial,"BB&T FINANCIAL CORP|BB&T FUNDS|BB&T Financial, FSB",13839,2689463,"cik,fdic","cik_id_match,standardized_name_matching",[]
1485,dempsey bd|e trade bank,"DEMPSEY & CO LLC /BD|DEMPSEY & COMPANY, L.L.C.|E*TRADE CAPITAL MARKETS - EXECUTION SERVICES LLC|E*TRADE Bank",28051,370271,"cik,fdic","cik_id_match,fuzzy_matching",[73.6842105263158]
1486,dempsey|e trade bank,"DEMPSEY & CO LLC /BD|DEMPSEY & COMPANY, L.L.C.|E*TRADE CAPITAL MARKETS - EXECUTION SERVICES LLC|E*TRADE Bank",28051,370271,"cik,fdic","cik_id_match,fuzzy_matching",[73.6842105263158]
1487,e trade capital markets - execution services|e trade bank,"DEMPSEY & CO LLC /BD|DEMPSEY & COMPANY, L.L.C.|E*TRADE CAPITAL MARKETS - EXECUTION SERVICES LLC|E*TRADE Bank",28051,370271,"cik,fdic","cik_id_match,fuzzy_matching",[73.6842105263158]
3278,united california bank,BANK OF THE WEST|SANWA BANK CALIFORNIA|UNITED CALIFORNIA BANK|UNITED CALIFORNIA BANK,59951,438368,"cik,fdic","cik_id_match,standardized_name_matching",[]
...,...,...,...,...,...,...,...
765521,morgan stanley bank,"MORGAN STANLEY BANK, N.A.|Morgan Stanley Bank, National Association",1871769,1456501,"cik,fdic",standardized_name_matching,[]
791298,midwest heritage bank,"MIDWEST HERITAGE BANK, FSB|Midwest Heritage Bank, FSB",1906805,203043,"cik,fdic",standardized_name_matching,[]
792809,merchants bank indiana,MERCHANTS BANK OF INDIANA|Merchants Bank of Indiana,1908863,963945,"cik,fdic",standardized_name_matching,[]
796298,central bank brazil|central bank boone,CENTRAL BANK OF BRAZIL|CENTRAL BANK OF BOONE COUNTY,1914083,299046,"cik,fdic",fuzzy_matching,[81.08108108108108]


In [278]:
remaining_fdic_df

,FED_RSSD,standardized_names,aliases,sources,matching_type
0,0,american savings loan|mckinley savings loan|united first savings loan|cheltenham savings loan|liberty savings|home bank florida|pine belt savings loan|concho valley savings loan|colonial savings loan|first savings bank|sterling savings|barbary coast savings bank|sequoia savings bank|citizens savings loan|truman savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
1,28,1st atlantic bank,1st Atlantic Bank,fdic,NaN
2,37,bank hancock,Bank of Hancock County,fdic,NaN
3,46,republic bank,Republic Bank,fdic,NaN
4,55,bank littlefork,State Bank of Littlefork,fdic,NaN
...,...,...,...,...,...
24716,4536084,bank bird--hand,Bank of Bird-in-Hand,fdic,NaN
24717,4569167,new traditions bank,New Traditions Bank,fdic,NaN
24718,4845861,primary bank,Primary Bank,fdic,NaN
24719,5050028,international bank,International Bank of Commerce,fdic,NaN


In [279]:
enriched_fdic_df

,FED_RSSD,standardized_names,aliases,sources,matching_type
0,0,american savings loan|mckinley savings loan|united first savings loan|cheltenham savings loan|liberty savings|home bank florida|pine belt savings loan|concho valley savings loan|colonial savings loan|first savings bank|sterling savings|barbary coast savings bank|sequoia savings bank|citizens savings loan|truman savings loan,"American Savings and Loan Association|McKinley Federal Savings and Loan Association|United First Federal Savings and Loan Association|Cheltenham Federal Savings and Loan Association|Liberty Savings Association|Home Federal Bank of Florida, F.S.B.|Pine Belt Federal Savings and Loan Association|Concho Valley Savings and Loan Association|Colonial Savings and Loan Association|First City Federal Savings Bank|Sterling Savings Association|Barbary Coast Savings Bank, FSB|Sequoia Savings Bank, FSB|Citizens Savings and Loan Association, A Federal Savings and Loan Associ|Truman Savings and Loan Association",fdic,FED_RSSD_match
1,28,1st atlantic bank,1st Atlantic Bank,fdic,NaN
2,37,bank hancock,Bank of Hancock County,fdic,NaN
3,46,republic bank,Republic Bank,fdic,NaN
4,55,bank littlefork,State Bank of Littlefork,fdic,NaN
...,...,...,...,...,...
24716,4536084,bank bird--hand,Bank of Bird-in-Hand,fdic,NaN
24717,4569167,new traditions bank,New Traditions Bank,fdic,NaN
24718,4845861,primary bank,Primary Bank,fdic,NaN
24719,5050028,international bank,International Bank of Commerce,fdic,NaN


In [280]:
final_crosswalk_df

,standardized_names,aliases,cik,FED_RSSD,sources,matching_type,fuzzy_matching_score
0,defined asset funds municipal invt tr fd new york ser 33,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD NEW YORK SER 33,3,<NA>,cik,NaN,[]
1,corporate income fund seventy ninth short term series,CORPORATE INCOME FUND SEVENTY NINTH SHORT TERM SERIES,13,<NA>,cik,NaN,[]
2,defined asset funds municipal invt tr fd mon pymt ser 155,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 155,14,<NA>,cik,NaN,[]
3,defined asset funds municipal invt tr fd mon pymt ser 156,DEFINED ASSET FUNDS MUNICIPAL INVT TR FD MON PYMT SER 156,17,<NA>,cik,NaN,[]
4,nuveen tax exempt unit trust series 169 national trust 169,NUVEEN TAX EXEMPT UNIT TRUST SERIES 169 NATIONAL TRUST 169,18,<NA>,cik,NaN,[]
...,...,...,...,...,...,...,...
834498,bank bird--hand,Bank of Bird-in-Hand,NaN,4536084,fdic,NaN,NaN
834499,new traditions bank,New Traditions Bank,NaN,4569167,fdic,NaN,NaN
834500,primary bank,Primary Bank,NaN,4845861,fdic,NaN,NaN
834501,international bank,International Bank of Commerce,NaN,5050028,fdic,NaN,NaN


In [281]:
# # Imported function from the regextable-python repository
# # get_match_candidate_score get the matching score between two names
# def get_match_candidate_score(frequency_dict, org_name, candidate_match_name):
#     if not isinstance(org_name, str):
#         org_name = ""
#     if not isinstance(candidate_match_name, str):
#         candidate_match_name = ""
    
#     if not org_name or not candidate_match_name:
#         return 0.0
    
#     org_tokens = org_name.split(' ')
    
#     # tokenize the candidate match
#     candidate_match_tokens = set(candidate_match_name.split(" "))

#     max_dist = 1

#     # Calculate the match score
#     total_inverse_frequency = 0
#     total_matching_inverse_frequency = 0
#     tokenized_name = org_tokens
#     for token in tokenized_name:
#         token_frequency = frequency_dict.get(token, 999999) # if token not found, give high frequency to ignore it
#         token_inverse_frequency = 1.0/token_frequency
#         total_inverse_frequency += token_inverse_frequency

#         best_token_similarity = 0.0
        
#         for candidate_token in candidate_match_tokens:
#             if not token or not candidate_token:
#                 continue

#             dist = damerau_levenshtein_distance(token, candidate_token)

#             if dist <= max_dist:
#                 max_len = max(len(token), len(candidate_token))
#                 if max_len == 0: continue
#                 similarity = 1.0 - (dist / max_len)

#                 best_token_similarity = max(best_token_similarity, similarity)
#         if best_token_similarity > 0.0:
#             total_matching_inverse_frequency += token_inverse_frequency * best_token_similarity
    
#     try:
#         match_score = total_matching_inverse_frequency / total_inverse_frequency
#     except ZeroDivisionError:
#         match_score = 0.0
    
#     #Multiplicative DL Penalty
#     m = len(org_name)
#     n = len(candidate_match_name)
#     if m == 0 or n == 0: return 0.0

#     dl_distance = damerau_levenshtein_distance(org_name, candidate_match_name)
#     normalized_dl = dl_distance / max(m, n)

#     final_score = match_score * (1 - normalized_dl)
#     return max(0.0, final_score)
        
        
# def clean_match_score(x):
#     if x is None or x is np.nan or pd.isnull(x) or x == "":
#         return np.nan
#     elif isinstance(x, str) and not x.isnumeric():
#         unit_multiplier = 1
#         if "B" in x:
#             x = x[:-1]
#             unit_multiplier = 1000000000
#         if "M" in x:
#             x = x[:-1]
#             unit_multiplier = 1000000
#         if "K" in x:
#             x = x[:-1]
#             unit_multiplier = 1000
#         x = x.replace(",", "")
#         try:
#             x = float(x) * unit_multiplier
#             return x
#         except:
#             return np.nan
#     else:
#         return float(x)

# from collections import defaultdict

# frequency_df = pd.DataFrame()
# frequency_df['std_name'] = unmatched_for_fuzzy['std_name']
# frequency_dict = defaultdict(int)

# # Iterate over the clean organization names
# for name in frequency_df['std_name']:
#     # Split the name into tokens (words)
#     tokens = name.split(' ')

#     # Update the count for each token
#     for token in tokens:
#         # Filter out empty strings that might result from extra spaces
#         if token:
#             frequency_dict[token] += 1

# # Convert defaultdict back to a standard dict for the function
# frequency_data = dict(frequency_dict)

# print("Printing the frequency_dict")
# print(frequency_data)
# from union_find import UnionFind

# # initialize union-find here
# uf = UnionFind(len(unmatched_for_fuzzy)) 
# print(len(uf.parent))

# # uf.unite(1, 1000)
# # uf.unite(1, 30)
# # set_i = uf.find(1000)
# # set_j = uf.find(1)
# # set_k = uf.find(30)

# # print(f'set_i: {set_i}, set_j: {set_j}, set_k: {set_k}')
# # print(uf.find(200))
# from rapidfuzz import process, fuzz
# from rapidfuzz.distance import JaroWinkler
# # score = fuzz.token_set_ratio("bank of american", 'bank of america')
# # score

# # score = JaroWinkler.similarity("bank of american", "bank of amercia")
# # score
# # count = 0
# for i in range(len(unmatched_for_fuzzy)):
#    # count += 1 
#    # if count % 10 == 0:
#    #    print(count)
      
#    item_i = unmatched_for_fuzzy.iloc[i]
#    for j in range(i+1, len(unmatched_for_fuzzy)):
#       item_j = unmatched_for_fuzzy.iloc[j]
      
#       # add union-find here
#       set_i = uf.find(i)
#       set_j = uf.find(j)
#       # When two index are in the same set, they already matched, skip
#       if set_i == set_j:
#          continue 
      
#       # score = get_match_candidate_score(frequency_dict, item_i['std_name'], item_j['std_name']) * 100
#       score = fuzz.token_set_ratio(item_i['std_name'], item_j['std_name'])
#       # score = JaroWinkler.similarity(item_i['std_name'], item_j['std_name']) * 100
#       # if j % 3 == 0: 
#       #    print(f"i is {i} and j is {j}, score is {score}")
      
#       if score < 93: 
#          # print(f"Warn — itemj name: {item_j['std_name']}, itemi name: {item_i['std_name']}")
#          continue
      
#       uf.unite(i, j)
#       # print(f"Match: i is {i} and j is {j}")
#       # print(f"item_i: {item_i['std_name']}, item_j: {item_j['std_name']}")
      
      
#    if i== 50:
#       break

# # The dictionary of parents and their row lists so that indices in the same list will be turned into a series
# # that will be merged into the new matched_df
# index_dict = defaultdict(list)
# for i in range(len(uf.parent)):
#     current_parent = uf.find(i)
#     index_dict[current_parent].append(i)
# index_dict = dict(index_dict)
# print(index_dict)
# print(len(index_dict))


# # TODO: Implement the for-loop that goes through all the keys in the sets and appends a new series to matched_df
# cols = ['standardized_names', 'aliases', 'cik', 'sources', 'original_index', 'score']
# matched_df = pd.DataFrame(columns = cols)

# count = 0
# for key in index_dict:
#     count += 1 
#     if count % 1000 == 0:
#         print(count)
#     row_list = index_dict[key]
#     new_std_name = ""
#     new_aliases = ""
#     new_sources = ""
#     new_cik = []
#     new_original_indices = []
    
#     for i in range(len(row_list)):
#         row = row_list[i]
#         if i == len(row_list) - 1:
#             new_std_name += unmatched_for_fuzzy.iloc[row]['std_name']
#             new_aliases += unmatched_for_fuzzy.iloc[row]['raw_name']
#             new_sources += unmatched_for_fuzzy.iloc[row]['source']
#             new_cik.append(unmatched_for_fuzzy.iloc[row]['cik'])
#             new_original_indices.append(row)
#         else:
#             new_std_name += unmatched_for_fuzzy.iloc[row]['std_name'] + "|"
#             new_aliases += unmatched_for_fuzzy.iloc[row]['raw_name'] + "|"
#             new_sources += unmatched_for_fuzzy.iloc[row]['source'] + ","
#             new_cik.append(unmatched_for_fuzzy.iloc[row]['cik']) 
#             new_original_indices.append(row)  
        
#         combined_row_dict = {"standardized_names": new_std_name,
#                              "aliases": new_aliases,
#                              "sources": new_sources,
#                              "cik": new_cik,
#                              "original_indices": new_original_indices} 
#         new_row_series = pd.Series(combined_row_dict)
#         matched_df = pd.concat([matched_df, new_row_series.to_frame().T])
        
# matched_df
          